In [6]:
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from scipy.stats import pearsonr

# Funzione per il fit lineare
def linear_fit(x, a, b):
    return a * x + b

# Carica il file CSV
file_path = r'ERA_PV_stazioni.csv'  # Modifica con il tuo percorso del file
df = pd.read_csv(file_path, delimiter=',', decimal='.')

# Converti la colonna 'data' in formato datetime
df['data'] = pd.to_datetime(df['data'], format='%Y-%m-%d')

# Aggrega i dati giornalieri per ogni 'nome'
df_daily = df.groupby(['nome', 'data']).agg({'rad': 'sum', 'shortwave_radiation': 'sum'}).reset_index()

# Filtra i dati per rimuovere i giorni con radiazione nulla
df_daily = df_daily[(df_daily['rad'] > 0) & (df_daily['shortwave_radiation'] > 0)]

# Lista per memorizzare i risultati
results = []

# Creiamo il fit lineare per ogni 'nome'
unique_names = df_daily['nome'].unique()

for name in unique_names:
    df_name = df_daily[df_daily['nome'] == name]
    
    # Moltiplica i valori dell'asse x (shortwave_radiation) per 3.6
    x = df_name['shortwave_radiation'] * 3.6
    y = df_name['rad']

    # Controlla se ci sono valori NaN o infiniti
    if x.isna().any() or y.isna().any():
        print(f"Ci sono valori NaN in x o y per '{name}'.")
        continue
    if np.isinf(x).any() or np.isinf(y).any():
        print(f"Ci sono valori infiniti in x o y per '{name}'.")
        continue

    # Calcola il fit e i parametri
    try:
        popt, _ = curve_fit(linear_fit, x, y, p0=[1, 0], maxfev=10000)
        a, b = popt  # a = pendenza, b = intercetta
        
        # Calcola il coefficiente di determinazione R^2
        residuals = y - linear_fit(x, *popt)
        ss_res = np.sum(residuals**2)
        ss_tot = np.sum((y - np.mean(y))**2)
        r_squared = 1 - (ss_res / ss_tot)
        
        # Aggiungi i risultati alla lista
        results.append({'name': name, 'r^2': r_squared, 'pendenza': a})
    
    except RuntimeError as e:
        print(f"Errore nel fitting per '{name}': {e}")

# Crea un DataFrame con i risultati
df_results = pd.DataFrame(results, columns=['name', 'r^2', 'pendenza'])

# Stampa i risultati
print(df_results)


                   name       r^2  pendenza
0                  ALBA  0.903638  1.065754
1            ALESSANDRI  0.927220  1.081755
2                  ASTI  0.899673  1.070913
3             AVIGLIANA  0.893588  1.000931
4             BASALUZZO  0.915761  1.084813
5   BORGOFRANCO_D'IVREA  0.845798  0.930611
6           BORGOMANERO  0.920239  1.052590
7                   BRA  0.894043  1.022679
8    BUTTIGLIERA_D'ASTI  0.870007  1.049297
9       CANDIA_CANAVESE  0.914214  1.028782
10           CARMAGNOLA  0.892139  1.028314
11    CASALE_MONFERRATO  0.919967  1.110767
12  COSTIGLIOLE_SALUZZO  0.882814  1.009656
13              CUMIANA  0.883518  1.014902
14                CUNEO  0.892629  1.012062
15              FOSSANO  0.871669  1.016682
16   ISOLA_SANT'ANTONIO  0.923711  1.067238
17           MOMBALDONE  0.864420  0.987145
18           MOMBARCARO  0.838902  0.905867
19           MONCALIERI  0.899527  1.066699
20          NOVI_LIGURE  0.911361  1.100476
21        PINO_TORINESE  0.80677